# BandList.fit — spectral parameter recovery test

In [2]:
%run pylib/tools dark
from utilities.ipynb_docgen import show, show_date
from pylib.tools import np
from importlib import reload
from pylib.psf_func import PSFlist
from like3 import bands as b; reload(b)
from like3.bands import BandList
from like3 import sourcelist as sl; reload(sl)
from like3.sourcelist import SourceModel

show_date()


<h5 style="text-align:right; margin-right:15px"> 2026-03-29 08:49</h5>

## Setup: simulate data with known parameters

In [3]:
# Build source model and BandList with PSF
source_model = SourceModel.demo()
df = PSFlist.demo_df()
df['nside'] = BandList.nsides

bandlist = BandList(source_model, df)
bandlist.simulate(random_state=42)

# Record the true parameter values before any perturbation
true_params = source_model.parameters.get_parameters().copy()
true_names  = source_model.parameter_names
show(f'**True parameters:** {dict(zip(true_names, true_params.round(4)))}')


Model: SourceModel: 2 sources with 4 free parameters


**True parameters:** {'Pulsar_Norm': -11.0, 'Pulsar_Index': 2.0, 'Blazar_Norm': -11.3979, 'Blazar_Index': 2.0}

## Fit: perturb parameters then recover with BandList.fit()

In [25]:
# Perturb all free parameters by 20% to give the fitter something to recover from
perturbed = true_params * 1.0
source_model.parameters.set_parameters(perturbed)
show(f'**Perturbed parameters:** {dict(zip(true_names, perturbed.round(4)))}')

# Run the fit (simplex by default)
fitvalue, best_pars, errors = bandlist.fit(quiet=False)

show(f"""
**Fit result** (log-likelihood  change = {fitvalue:.1f})

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {r:.4f} | {e:.4f} | {(r - t) / e if e > 0 else np.nan:.1f} |"
    for n, t, r, e in zip(true_names, true_params, best_pars, errors)
))


**Perturbed parameters:** {'Pulsar_Norm': -11.0, 'Pulsar_Index': 2.0, 'Blazar_Norm': -11.3979, 'Blazar_Index': 2.0}

using optimize.fmin_l_bfgs_b with parameter bounds [[-15.  -3.]
 [ -5.   5.]
 [-17.  -3.]
 [ -5.   5.]]
, kw= {}
{'grad': array([ 7.23443409e-05,  2.16473581e-04,  1.84469130e-04, -1.86606648e-04]), 'task': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 15, 'nit': 9, 'warnflag': 0}
Function value at minimum: -12.748185
Attempting to invert full hessian...


**Fit result** (log-likelihood  change = -12.7)

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
| Pulsar_Norm | -11.0000 | -10.9838 | 0.0060 | 2.7 |
| Pulsar_Index | 2.0000 | 1.9842 | 0.0089 | -1.8 |
| Blazar_Norm | -11.3979 | -11.3619 | 0.0092 | 3.9 |
| Blazar_Index | 2.0000 | 1.9412 | 0.0144 | -4.1 |

## Band selection: fit using only a subset of bands

In [26]:
# Perturb again
source_model.parameters.set_parameters(true_params * 1.2)

# Fit using only the 6 middle bands (indices 3-8)
mid_bands = list(range(3, 9))
bandlist.select(mid_bands)
show(f'Fitting with bands {mid_bands}: {[bandlist[i] for i in mid_bands]}')

fitvalue_mid, best_pars_mid, errors_mid = bandlist.fit(quiet=True)

# Reset to all bands
bandlist.select()

show(f"""
**Subset-band fit result** (negative log-likelihood = {fitvalue_mid:.1f})

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {r:.4f} | {e:.4f} | {(r - t) / e if e > 0 else np.nan:.1f} |"
    for n, t, r, e in zip(true_names, true_params, best_pars_mid, errors_mid)
))


Fitting with bands [3, 4, 5, 6, 7, 8]: [Band(energy=750.0 MeV, et=0 nside=128), Band(energy=1334.0 MeV, et=0 nside=256), Band(energy=2371.0 MeV, et=0 nside=512), Band(energy=4217.0 MeV, et=0 nside=512), Band(energy=7499.0 MeV, et=0 nside=512), Band(energy=13335.0 MeV, et=0 nside=1024)]

**Subset-band fit result** (negative log-likelihood = -26474.2)

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
| Pulsar_Norm | -11.0000 | -10.9813 | 0.0063 | 3.0 |
| Pulsar_Index | 2.0000 | 1.9617 | 0.0243 | -1.6 |
| Blazar_Norm | -11.3979 | -11.3537 | 0.0140 | 3.2 |
| Blazar_Index | 2.0000 | 1.9824 | 0.0398 | -0.4 |

In [27]:
from time import perf_counter
import numpy as np

# Compare fit runtime with and without analytic gradient from the same start point.
def timed_fit(use_gradient, start_pars):
    source_model.parameters.set_parameters(start_pars.copy())
    t0 = perf_counter()
    out = bandlist.fit(method='l-bfgs-b', use_gradient=use_gradient, quiet=True)
    dt = perf_counter() - t0
    return dt, out

n_trials = 3
start_pars = true_params * 1.2

# Ensure all bands are active for a fair comparison.
bandlist.select()

times_no_grad = []
times_with_grad = []

for _ in range(n_trials):
    dt, _ = timed_fit(False, start_pars)
    times_no_grad.append(dt)

for _ in range(n_trials):
    dt, _ = timed_fit(True, start_pars)
    times_with_grad.append(dt)

mean_no_grad = float(np.mean(times_no_grad))
mean_with_grad = float(np.mean(times_with_grad))
speedup = mean_no_grad / mean_with_grad if mean_with_grad > 0 else np.nan

print('BandList.fit timing comparison (method=l-bfgs-b)')
print(f'  trials: {n_trials}')
print(f'  no gradient : {mean_no_grad:.3f} s  (runs={np.round(times_no_grad, 3)})')
print(f'  with gradient: {mean_with_grad:.3f} s  (runs={np.round(times_with_grad, 3)})')
print(f'  speedup (no_grad / grad): {speedup:.1f}x')

BandList.fit timing comparison (method=l-bfgs-b)
  trials: 3
  no gradient : 14.265 s  (runs=[14.683 14.109 14.004])
  with gradient: 3.540 s  (runs=[3.532 3.549 3.539])
  speedup (no_grad / grad): 4.0x


In [4]:
from time import perf_counter
import numpy as np

bandlist.select()  # all bands

configs = [
    ('simplex',  False),
    ('powell',   False),
    ('l-bfgs-b', False),
    ('l-bfgs-b', True),
]
n_reps = 2  # repeat each to reduce noise

rows = []
for method, use_grad in configs:
    label = f'{method}{"  +grad" if use_grad else ""}'
    times = []
    for _ in range(n_reps):
        source_model.parameters.set_parameters(true_params * 1.2)
        t0 = perf_counter()
        # estimate_errors=False: only measure optimizer time, not Hessian cost
        fval, pars = bandlist.fit(method=method, use_gradient=use_grad,
                                  quiet=True, estimate_errors=False)
        times.append(perf_counter() - t0)
    mean_t = float(np.mean(times))
    residuals = np.abs(pars - true_params)
    rows.append((label, mean_t, float(residuals.max())))

# Sort fastest first
rows.sort(key=lambda x: x[1])
fastest = rows[0][1]

show(f"""
## Method speed comparison ({n_reps} reps each, all 12 bands)

| Method | Time (s) | Rel time | Max |\u0394par| |
|--------|:--------:|:--------:|:----------:|
""" + "\n".join(
    f"| {label} | {t:.2f} | {t/fastest:.2f}x | {maxd:.5f} |"
    for label, t, maxd in rows
))

## Method speed comparison (2 reps each, all 12 bands)

| Method | Time (s) | Rel time | Max |Δpar| |
|--------|:--------:|:--------:|:----------:|
| powell | 1.74 | 1.00x | 0.36160 |
| l-bfgs-b  +grad | 1.98 | 1.14x | 0.05882 |
| l-bfgs-b | 12.06 | 6.92x | 0.05882 |
| simplex | 12.37 | 7.09x | 0.05882 |

## Variance vs predicted errors: ensemble of simulations

In [28]:
n_sim = 50
recovered = []
predicted_errors = []

for seed in range(n_sim):
    # Set true parameters BEFORE simulating so each realisation comes from the true model.
    source_model.parameters.set_parameters(true_params.copy())
    bandlist.simulate(random_state=seed)
    # Start the fit from the true parameters as well.
    source_model.parameters.set_parameters(true_params.copy())
    fitvalue_sim, best_sim, err_sim = bandlist.fit(use_gradient=True, quiet=True)
    recovered.append(best_sim)
    predicted_errors.append(err_sim)

recovered = np.array(recovered)          # (n_sim, n_par)
predicted_errors = np.array(predicted_errors)  # (n_sim, n_par)

obs_std   = recovered.std(axis=0)
pred_mean = predicted_errors.mean(axis=0)
bias      = recovered.mean(axis=0) - true_params

show(f"""
## Ensemble fit validation  (n = {n_sim} simulations)

| Parameter | True | Bias | Obs σ | Predicted σ | Obs/Pred |
|-----------|:----:|:----:|:-----:|:-----------:|:--------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {b:.4f} | {o:.4f} | {p:.4f} | {o/p:.3f} |"
    for n, t, b, o, p in zip(true_names, true_params, bias, obs_std, pred_mean)
))

## Ensemble fit validation  (n = 50 simulations)

| Parameter | True | Bias | Obs σ | Predicted σ | Obs/Pred |
|-----------|:----:|:----:|:-----:|:-----------:|:--------:|
| Pulsar_Norm | -11.0000 | 0.0134 | 0.0109 | 0.0060 | 1.817 |
| Pulsar_Index | 2.0000 | -0.0146 | 0.0080 | 0.0089 | 0.906 |
| Blazar_Norm | -11.3979 | 0.0388 | 0.0112 | 0.0092 | 1.212 |
| Blazar_Index | 2.0000 | -0.0576 | 0.0169 | 0.0144 | 1.177 |